# 13. External Validation

This notebook captures the additive external-validation layer added on top of the original REDNET-ML workflow. It does not change training labels, detector training, fusion training, benchmark outputs, or viewer logic.

The purpose is to make the validation extension reproducible and to keep the claim boundary explicit.

## Validation modes

REDNET-ML now supports three related but distinct validation modes:

- `event_validation`: external public or official event concordance
- `regional_advisory_validation`: lower-precision regional or operational advisories with wider timing windows
- `insitu_validation`: true field-sample validation when genuine in-situ records exist

Public reports, ministry notices, literature event records, and advisories are not in-situ measurements. They support external event concordance only.

## Safe and unsafe claims

Safe claims:

- The original REDNET-ML benchmark remains proxy-supervised internal validation.
- An additive external validation framework was implemented.
- Public-source event validation supports event-level external concordance analysis.
- True in-situ validation remains contingent on access to genuine field observations.
- The public event setup is supplementary and does not replace ecological field validation.
- Where plant-local public events are unavailable, a weaker regional advisory alignment layer can be used with wider timing windows.

Unsafe claims:

- In-situ validation was performed.
- The model was externally validated against field samples.
- Public reports were used as in-situ data.
- The external event setup proves ecological truth detection.
- All AOIs are equally well validated.

## Seeded inputs

The repository includes seeded inputs under `data/external_validation/`:

- `oman_hab_events_seed.csv`
- `oman_hab_events.csv`
- `oman_regional_advisories_seed.csv`
- `oman_regional_advisories.csv`
- `location_to_plant_map.csv`
- `insitu_records.csv`

The in-situ table is a schema placeholder by default. No true field records are bundled in the repository.

In [2]:
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'runs').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root from the current notebook working directory.')


repo = find_repo_root(Path.cwd().resolve())
data_dir = repo / 'data' / 'external_validation'
runs_dir = repo / 'runs' / 'eval' / 'external_validation'

print('Repo root:', repo)
print('Seeded inputs:')
for path in sorted(data_dir.glob('*')):
    print(' -', path.relative_to(repo))

print('\nOutput directory exists:', runs_dir.exists())
if runs_dir.exists():
    print('Run outputs:')
    for path in sorted(runs_dir.glob('*')):
        print(' -', path.relative_to(repo))


Repo root: /Users/ameerfiras/REDNET-ML
Seeded inputs:
 - data/external_validation/insitu_records.csv
 - data/external_validation/location_to_plant_map.csv
 - data/external_validation/oman_hab_events.csv
 - data/external_validation/oman_hab_events_seed.csv
 - data/external_validation/oman_regional_advisories.csv
 - data/external_validation/oman_regional_advisories_seed.csv

Output directory exists: True
Run outputs:
 - runs/eval/external_validation/aoi_event_validation_by_plant.csv
 - runs/eval/external_validation/aoi_event_validation_overview.md
 - runs/eval/external_validation/combined_aoi_validation_coverage.csv
 - runs/eval/external_validation/combined_aoi_validation_coverage.md
 - runs/eval/external_validation/event_lead_lag_table.csv
 - runs/eval/external_validation/event_validation_report.md
 - runs/eval/external_validation/event_validation_summary.json
 - runs/eval/external_validation/event_vs_nonevent_hab_prob.png
 - runs/eval/external_validation/event_vs_nonevent_score.png
 - 

## Event-validation workflow

Build the canonical public-event table:

```bash
python scripts/validation/build_external_events_table.py \
  --in_csv data/external_validation/oman_hab_events_seed.csv \
  --out_csv data/external_validation/oman_hab_events.csv
```

Match dated public events to plant prediction series:

```bash
python scripts/validation/match_external_validation.py \
  --mode event \
  --events_csv data/external_validation/oman_hab_events.csv \
  --out_csv runs/eval/external_validation/matched_external_events.csv
```

Score the matched event set:

```bash
python scripts/validation/score_external_validation.py \
  --mode event \
  --matched_csv runs/eval/external_validation/matched_external_events.csv \
  --outdir runs/eval/external_validation
```

## Regional advisory coverage

This weaker layer is only for wider AOI support when plant-local public event rows are sparse. It should be described as supporting evidence, not plant-day confirmation.

```bash
python scripts/validation/build_external_events_table.py \
  --in_csv data/external_validation/oman_regional_advisories_seed.csv \
  --out_csv data/external_validation/oman_regional_advisories.csv

python scripts/validation/match_external_validation.py \
  --mode event \
  --events_csv data/external_validation/oman_regional_advisories.csv \
  --primary_window_days 14 \
  --sensitivity_window_days 14 \
  --out_csv runs/eval/external_validation/regional_advisories/matched_external_events.csv

python scripts/validation/score_external_validation.py \
  --mode event \
  --matched_csv runs/eval/external_validation/regional_advisories/matched_external_events.csv \
  --outdir runs/eval/external_validation/regional_advisories
```

## In-situ pathway

The in-situ path exists as a schema and scoring workflow, but it remains inactive until genuine field records are added to `data/external_validation/insitu_records.csv`.

Expected columns:

- `sample_id`
- `datetime_utc`
- `lat`
- `lon`
- `plant_id`
- `hab_event`
- `evidence_type`
- `species`
- `cell_count`
- `toxin_value`
- `notes`
- `source_url`

Once true field records exist, run:

```bash
python scripts/validation/match_external_validation.py \
  --mode insitu \
  --insitu_csv data/external_validation/insitu_records.csv \
  --out_csv runs/eval/external_validation/matched_insitu_records.csv

python scripts/validation/score_external_validation.py \
  --mode insitu \
  --matched_csv runs/eval/external_validation/matched_insitu_records.csv \
  --outdir runs/eval/external_validation
```

In [1]:
import json
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'runs').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root from the current notebook working directory.')


repo = find_repo_root(Path.cwd().resolve())
summary_path = repo / 'runs' / 'eval' / 'external_validation' / 'event_validation_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    for key, value in summary.items():
        print(f'{key}: {value}')
else:
    print(f'No event-validation summary found at {summary_path}.')


mode: event_validation
input_csv: runs/eval/external_validation/matched_external_events.csv
watch_threshold: 0.55
action_threshold: 0.6238688594
n_rows: 8
primary_score_col: matched_ops_risk
n_primary_matched: 6
n_sensitivity_matched: 7
n_positive_primary: 2
n_negative_primary: 4
event_hit_rate_watch: 0.0
event_hit_rate_action: 0.0
median_positive_score: 0.46509360081533185
median_negative_score: 0.4041266214489295
median_positive_hab_prob: 0.38187308609463655
median_negative_hab_prob: 0.3916913907524203
auroc: 0.875
auprc: 0.8333333333333333
n: 6
event_window_median_score: 0.4650936008153319
nonevent_window_median_score: 0.4390968267366891
note: Primary external event validation uses matched plant-date windows only. This is event-based external concordance, not in-situ validation.


## Output files

Primary outputs are written under `runs/eval/external_validation/`:

- `matched_external_events.csv`
- `matched_insitu_records.csv`
- `event_validation_summary.json`
- `insitu_validation_summary.json`
- `event_validation_report.md`
- `top_ranked_event_matches.csv`
- `event_lead_lag_table.csv`
- `event_vs_nonevent_hab_prob.png` when enough matched windows exist

The original internal benchmark remains the main validation result. The external layer is additive and should be interpreted accordingly.